In [ ]:
import sys
import os

from opt_einsum import contract
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from visualisation import *
from plotly.subplots import make_subplots
from diffusion_geometry.classes.tensors import Function, VectorField, Form
from generate_data import gen_2d_data
from diffusion_geometry.classes.main import DiffusionGeometry
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from matplotlib.path import Path

In [ ]:
def sample_fibonacci_sphere(n: int, radius: float = 1.0) -> np.ndarray:
    indices = np.arange(n)
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle

    y = 1 - 2 * indices / (n - 1)
    r = np.sqrt(1 - y * y)
    theta = phi * indices

    x = np.cos(theta) * r
    z = np.sin(theta) * r

    points = radius * np.column_stack((x, y, z))
    return points


def spherical_frame(points: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Normalize points to unit sphere
    n = points / np.linalg.norm(points, axis=1, keepdims=True)
    x, y, z = n[:, 0], n[:, 1], n[:, 2]

    # e_phi = derivative wrt azimuthal angle φ
    e_phi = np.column_stack((-y, x, np.zeros_like(z)))
    norm_phi = np.linalg.norm(e_phi, axis=1, keepdims=True)
    # handle poles robustly
    norm_phi[norm_phi[:, 0] == 0] = 1.0
    e_phi /= norm_phi

    # e_theta = e_phi × n (right-handed frame)
    e_theta = np.cross(e_phi, n)

    return e_theta, e_phi, n



def sample_torus(n_theta: int, n_phi: int, R: float = 2.0, r: float = 0.7) -> np.ndarray:
    theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    theta, phi = np.meshgrid(theta, phi, indexing='ij')

    x = (R + r * np.cos(theta)) * np.cos(phi)
    y = (R + r * np.cos(theta)) * np.sin(phi)
    z = r * np.sin(theta)

    data = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    return data, theta.ravel(), phi.ravel()


def torus_frame(theta: np.ndarray, phi: np.ndarray, R: float, r: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Tangent directions
    e_theta = np.column_stack((
        -np.sin(theta) * np.cos(phi),
        -np.sin(theta) * np.sin(phi),
         np.cos(theta)
    ))

    e_phi = np.column_stack((
        -(R + r*np.cos(theta)) * np.sin(phi),
         (R + r*np.cos(theta)) * np.cos(phi),
         np.zeros_like(theta)
    ))

    # Normalize to get orthonormal frame
    e_theta /= np.linalg.norm(e_theta, axis=1, keepdims=True)
    e_phi /= np.linalg.norm(e_phi, axis=1, keepdims=True)

    # Normal vector (points outward from torus surface)
    n = np.column_stack((
        np.cos(theta) * np.cos(phi),
        np.cos(theta) * np.sin(phi),
        np.sin(theta)
    ))

    return e_theta, e_phi, n



def sample_hyperboloid_one_sheet(n_u: int, n_v: int,
                                 a: float = 1.0, 
                                 b: float = 1.0,
                                 c: float = 1.0,
                                 u_max: float = 1.2):

    u = np.linspace(-u_max, u_max, n_u)
    v = np.linspace(0, 2*np.pi, n_v, endpoint=False)
    u, v = np.meshgrid(u, v, indexing="ij")

    x = a * np.cosh(u) * np.cos(v)
    y = b * np.cosh(u) * np.sin(v)
    z = c * np.sinh(u)

    pts = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    return pts, u.ravel(), v.ravel()


def hyperboloid_frame(u: np.ndarray, v: np.ndarray,
                      a: float = 1.0,
                      b: float = 1.0,
                      c: float = 1.0):

    # First derivatives
    Xu = np.column_stack([
        a * np.sinh(u) * np.cos(v),
        b * np.sinh(u) * np.sin(v),
        c * np.cosh(u),
    ])

    Xv = np.column_stack([
        -a * np.cosh(u) * np.sin(v),
         b * np.cosh(u) * np.cos(v),
         np.zeros_like(u),
    ])

    # Normalize tangent vectors
    e_u = Xu / np.linalg.norm(Xu, axis=1, keepdims=True)
    e_v = Xv / np.linalg.norm(Xv, axis=1, keepdims=True)

    # Normal = e_u × e_v (right-handed)
    n = np.cross(e_u, e_v)
    n /= np.linalg.norm(n, axis=1, keepdims=True)

    return e_u, e_v, n



def plot_sphere(fig):
    data = sample_fibonacci_sphere(500, radius=1.0)
    dg = DiffusionGeometry.from_point_cloud(data)

    e_theta, e_phi, n = spherical_frame(data)

    e_theta = VectorField.from_pointwise_basis(e_theta, dg)
    e_phi = VectorField.from_pointwise_basis(e_phi, dg)

    curv = dg.sectional_curvature(e_theta, e_phi)
    fig1 = plot_scatter_3d(data, curv)
    fig.add_traces(list(fig1.data), rows=1, cols=1)


def plot_torus(fig):
    data, theta, phi = sample_torus(20, 40, R=2.0, r=0.7)
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100)

    e_theta, e_phi, n = torus_frame(theta, phi, R=2.0, r=0.7)

    e_theta = VectorField.from_pointwise_basis(e_theta, dg)
    e_phi = VectorField.from_pointwise_basis(e_phi, dg)

    curv = dg.sectional_curvature(e_theta, e_phi)
    fig1 = plot_scatter_3d(data, curv)
    fig.add_traces(list(fig1.data), rows=1, cols=2)

def plot_hyperboloid(fig):
    data, u, v = sample_hyperboloid_one_sheet(30, 40, a=1.0, b=1.0, c=1.0, u_max=1.2)
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100)

    e_u, e_v, n = hyperboloid_frame(u, v, a=1.0, b=1.0, c=1.0)

    e_u = VectorField.from_pointwise_basis(e_u, dg)
    e_v = VectorField.from_pointwise_basis(e_v, dg)

    curv = dg.sectional_curvature(e_u, e_v)
    fig1 = plot_scatter_3d(data, curv)
    fig.add_traces(list(fig1.data), rows=1, cols=3)
    

# ---------------
# Main Plot
# ---------------

num_rows = 1
num_cols = 3

specs = [
    [{"type": "scene"}]*num_cols,]


fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=False,
    shared_yaxes=False
)


# camera1 = dict( eye=dict(x=0, y=1.2, z=0.6),  
#     up=dict(x=0, y=0, z=1),     
#     center=dict(x=0, y=0, z=0))
# camera2 = dict( eye=dict(x=0, y=1.2, z=0.6),  
#     up=dict(x=0, y=0, z=1),     
#     center=dict(x=0, y=0, z=0))
# camera3 = dict( eye=dict(x=0, y=1.2, z=0.6),  
#     up=dict(x=0, y=0, z=1),     
#     center=dict(x=0, y=0, z=0))

cameras = [
    dict(eye=dict(x=1.2, y=0.0, z=0.6), up=dict(x=0, y=0, z=1)),
    dict(eye=dict(x=0.0, y=1.7, z=1), up=dict(x=0, y=0, z=1)),
    dict(eye=dict(x=-1.5, y=0.0, z=0.7), up=dict(x=0, y=0, z=1), center=dict(x=0, y=0, z=-0.1)),
]

# fig.update_layout(scene = dict(camera=camera1))
# fig.update_layout(scene2 = dict(camera=camera2))
# fig.update_layout(scene3 = dict(camera=camera3))

plot_sphere(fig)
plot_torus(fig)
plot_hyperboloid(fig)

for i, cam in enumerate(cameras, start=1):
    key = "scene" if i == 1 else f"scene{i}"
    fig.update_layout({key: dict(camera=cam, aspectmode="data")})

fig.update_layout(width=1200, height=400) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=10))
fig.show()

fig.write_image("figs/output.pdf")
